# TrustLens — Phase I: Unified Listing-Level Feature Store

**Consolidated Cross-Modal Marketplace Analytics Table**

This notebook consolidates the deterministic outputs of Phases B through H into a single canonical analytical table where **1 row = 1 canonical OLX listing**.

### Strict Scientific Guardrails
- **NO target variables**, fraud scores, scam probabilities, or risk rankings.
- **NO seller identity imputation**; missing seller data remains unobserved.
- **Explicit missingness**: 0 = observed absence, NULL = unbenchmarked or unevaluated.
- **Zero data leakage**: Purely observational marketplace variables for Phase J statistical anomaly analysis.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from trustlens.marketplace.feature_store import UnifiedFeatureStoreBuilder
print("Libraries and UnifiedFeatureStoreBuilder imported successfully!")

## 1. Load Canonical Base Listings (Phase A)
Inspect the authoritative canonical listing population.

In [ ]:
base_df = pq.read_table("data/olx_processed/listings.parquet").to_pandas()
print(f"Canonical listing population: {len(base_df):,} listings")
print(f"Unique listing_ids: {base_df.listing_id.nunique():,}")

## 2. Load Frozen Phase B–H Artifacts
Initialize builder and load all prior forensic layers strictly read-only.

In [ ]:
builder = UnifiedFeatureStoreBuilder()
builder.load_all_sources()
print(f"Normalized listings (Phase B): {len(builder.df_norm):,}")
print(f"Fingerprints (Phase C): {len(builder.df_fingerprints):,}")
print(f"Deep visual relationships (Phase D): {len(builder.df_deep_rel):,}")
print(f"Image OCR records (Phase E): {len(builder.df_ocr):,}")
print(f"Multimodal inconsistencies (Phase E): {len(builder.df_incons):,}")
print(f"Text features (Phase F): {len(builder.df_text):,}")
print(f"Image authenticity (Phase G): {len(builder.df_auth):,}")
print(f"AI detector results (Phase G.1): {len(builder.df_ai):,}")
print(f"Relationship features (Phase H): {len(builder.df_rel_feat):,}")

## 3. Build & Assemble Unified Feature Store
Construct deterministic listing-level features across all 9 families.

In [ ]:
df_unified = builder.build_unified_feature_store()
print(f"Dataset dimensions: {df_unified.shape[0]:,} rows x {df_unified.shape[1]:,} columns")
assert len(df_unified) == len(base_df)
assert df_unified.listing_id.nunique() == len(base_df)

## 4. Coverage & Availability Audit
Examine explicit availability indicators across all feature modalities.

In [ ]:
avail_cols = [c for c in df_unified.columns if "_available" in c]
cov_df = pd.DataFrame({
    "available_count": [df_unified[c].sum() for c in avail_cols],
    "coverage_share": [df_unified[c].mean() * 100 for c in avail_cols],
}, index=avail_cols)
print(cov_df.round(1))

## 5. Price Benchmarks & Discount Distribution (Phase B)
Inspect comparable product group benchmarking and price delta distribution.

In [ ]:
benchmarked = df_unified[df_unified.comparable_product_group.notnull()]
print(f"Benchmarked device listings: {len(benchmarked):,} ({len(benchmarked)/len(df_unified)*100:.1f}%)")
print(f"Listings <= -35% below group median: {benchmarked.price_below_35_pct_median_flag.sum():,} ({benchmarked.price_below_35_pct_median_flag.mean()*100:.1f}%)")
print(f"Listings <= -50% below group median: {benchmarked.price_below_50_pct_median_flag.sum():,} ({benchmarked.price_below_50_pct_median_flag.mean()*100:.1f}%)")

## 6. Multi-Modal Relationship & Inconsistency Confluence
Examine the cross-tabulation of media reuse, text similarity, and multimodal inconsistencies.

In [ ]:
print(f"Listings with exact image reuse: {(df_unified.exact_image_reuse_count > 0).sum():,}")
print(f"Listings with exact title reuse: {(df_unified.exact_title_reuse_count > 0).sum():,}")
print(f"Listings with multimodal inconsistencies: {df_unified.has_inconsistency_candidate.sum():,}")
print(f"Listings with AI detector candidate flags: {df_unified.ai_generation_candidate_flag.sum():,}")
print(f"Listings in non-singleton components: {(~df_unified.is_singleton_listing).sum():,}")

## 7. Exported Parquet Table Verification
Verify file existence and column count on disk.

In [ ]:
p_path = Path("data/olx_processed/unified_features.parquet")
assert p_path.exists()
p_table = pq.read_table(p_path)
print(f"Successfully verified {p_path} on disk: {p_table.num_rows:,} rows x {p_table.num_columns:,} columns")